# Exercises XP: Introduction to LLMs

Use this guided notebook to follow the platform instructions step by step. Prefilled cells are ready to run; cells containing **TODO** markers need your input.


## 👩‍🏫👩🏿‍🏫 What you’ll learn
- Understand what Large Language Models (LLMs) can do.
- Review the Transformer architecture and the tokenization pipeline.
- Differentiate between pretraining and fine-tuning.
- Generate text with a pretrained language model.


## 🛠️ What you will create
- Markdown answers describing key NLP concepts.
- Python code that loads GPT-2 (or a similar causal LM) and performs basic tokenization and generation.


> **Learning point**
> Work through the exercises sequentially. Run installation cells only once, then focus on filling each TODO before executing the corresponding code.


## 🌟 Exercise 1 · What are Large Language Models?


### 1.1 Define LLMs

**Large Language Models (LLMs)** are deep neural networks — almost always based on the
**Transformer** architecture — that are trained on enormous amounts of text (books, websites,
code, etc.). During training they learn the statistical patterns of language by repeatedly
predicting missing or next words. Because they have billions of parameters, they capture grammar,
facts, reasoning patterns and writing styles.

In plain words: an LLM is a system that, given some text, can predict what text should come next,
and it does this so well that it appears to "understand" and "produce" language.

**Tasks LLMs are designed to solve:**
- **Text generation** — writing, continuing or completing text.
- **Summarization** — condensing long documents into short summaries.
- **Translation** — converting text from one language to another.
- **Question answering** — responding to questions using learned knowledge.
- **Classification & sentiment analysis** — labeling text (e.g. positive/negative).
- **Code generation & explanation** — writing and describing source code.
- **Conversation / chat** — acting as assistants that hold a dialogue.


### 1.2 Prefilled · install core libraries
Run once to install `transformers`, `torch`, and supporting utilities exactly as in the enoncé.


In [ ]:
%pip install --quiet transformers matplotlib --upgrade


### 1.3 Load GPT for causal text generation
Reuse the snippet from the platform: declare the model name, tokenizer, and model weights.


In [ ]:
# Core imports: AutoTokenizer / AutoModelForCausalLM auto-detect the right classes
# for the chosen checkpoint, and `pipeline` gives a high-level helper for generation.
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")  # Hide noisy deprecation/usage warnings for a clean output.

# Name of the pretrained checkpoint to download from the Hugging Face Hub.
# 'gpt2' is a small, fast causal language model — ideal for learning.
model_name = "gpt2"

# The tokenizer turns text into token IDs the model understands (and back again).
tokenizer = AutoTokenizer.from_pretrained(model_name)

# The model holds the pretrained weights and predicts the next token in a sequence.
model = AutoModelForCausalLM.from_pretrained(model_name)

# Safety check: make sure none of the three required objects is left unset.
if None in (model_name, tokenizer, model):
    raise ValueError("Fill in model_name, tokenizer, and model before continuing.")

print(f"\nModel '{model_name}' loaded successfully!")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence.
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")


## 🌟 Exercise 2 · Transformer Architecture and Tokenization


**Tokenization (in my own words):**

A model cannot read raw text — it only works with numbers. **Tokenization** is the step that
splits a sentence into smaller pieces called **tokens** (whole words, parts of words/subwords, or
even single characters), and then maps each token to a unique integer **token ID** using the
model's vocabulary.

GPT-2 uses **Byte-Pair Encoding (BPE)**, a subword method: frequent words become a single token,
while rare words are broken into several subword tokens. This keeps the vocabulary at a manageable
size while still being able to represent any word. These token IDs are what we actually feed into
the Transformer, and after generation the IDs are converted back into readable text.


In [ ]:
# A short sentence to feed through the tokenizer.
text = "Large language models are transforming the way we work."
if text is None:
    raise ValueError("Define the variable `text` with a short sentence.")

# Split the sentence into subword tokens (strings) using GPT-2's BPE vocabulary.
tokens = tokenizer.tokenize(text)
# Map each token string to its unique integer ID from the model's vocabulary.
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")

# Axis labels and title for the bar chart below.
x_label = "Tokens"        # Each bar corresponds to one token.
y_label = "Token IDs"     # Bar height is the token's vocabulary ID.
title = "GPT-2 Token IDs for the input sentence"

if None in (x_label, y_label, title):
    raise ValueError("Set x_label, y_label, and title before plotting.")

# Visualize how each token maps to its numeric ID.
plt.figure(figsize=(10, 4))
plt.bar(tokens, token_ids, color="skyblue")
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.title(title)
plt.xticks(rotation=45)  # Rotate token labels so they don't overlap.
plt.show()


## 🌟 Exercise 3 · Token IDs and special prefixes


In [ ]:
# Make sure Exercise 2 has already produced `tokens` and `token_ids`.
if 'tokens' not in globals() or 'token_ids' not in globals():
    raise ValueError("Run Exercise 2 to define `tokens` and `token_ids` first.")

# Walk through both lists in parallel with zip() and print each token next to its numeric id.
print(f"{'Token':<15} | Token ID")
print("-" * 30)
for token, token_id in zip(tokens, token_ids):
    print(f"{token:<15} | {token_id}")


**What does the `Ġ` prefix mean?**

In GPT-2's byte-level BPE vocabulary, the special character **`Ġ`** represents a **leading space**
(a space that comes *before* the token). GPT-2 encodes spaces as part of the token itself instead
of treating them separately, so `Ġworld` means "<space>world" (the word *world* preceded by a
space), while `world` (no `Ġ`) would be the word at the very start of the text or attached to the
previous token.

This lets the tokenizer tell the difference between, for example, the start of a new word and a
word-piece in the middle of a word, and it allows the original text — spaces included — to be
reconstructed exactly when the tokens are decoded back.


## 🌟 Exercise 4 · Generate simple text


Create a fresh prompt, run the generator, and observe how the model extends your sentence token by token.


In [ ]:
# Build a high-level text-generation pipeline around our loaded model + tokenizer.
# device=0 uses the GPU if one is available, otherwise -1 falls back to the CPU.
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

# The prompt the model will continue/extend.
input_text = "In the future, artificial intelligence will"
if input_text is None:
    raise ValueError("Set `input_text` before generating.")

# Generation settings that control length and how "creative"/random the output is.
gen_kwargs = {
    "max_new_tokens": 60,  # How many new tokens to generate after the prompt.
    "temperature": 0.8,    # Higher = more random/creative, lower = more focused/deterministic.
    "top_p": 0.95,         # Nucleus sampling: keep the smallest set of tokens with cumulative prob 0.95.
    "do_sample": True,     # Enable sampling (required for temperature/top_p to take effect).
}

# Run the generator; it returns a list of dicts, one per generated sequence.
output_ids = generator(input_text, **gen_kwargs)
# Extract the generated text (prompt + continuation) from the first result.
output_text = output_ids[0]["generated_text"]

print(f"Input: {input_text}")
print(f"Generated Output: {output_text}")


> **Learning point**
> Compare the generated continuation with your expectations. Which knobs (temperature, max tokens) change the style the most?


Here's a summary that can help you decide of how to fix these parameters:

![image.png](https://github.com/user-attachments/assets/a4c444d7-fab8-4f56-b7c7-00a15900cb5a)